# 02 · Autograd and gradient descent

Every training loop you will ever write is the same four lines:

```python
loss = loss_fn(model(x), y)     # forward: build the computation graph, end in ONE number
optimizer.zero_grad()           # erase the gradients of the previous step
loss.backward()                 # backward: fill .grad of every parameter with d loss / d parameter
optimizer.step()                # move every parameter a little against its gradient
```

If you only memorise these lines you will be lost the moment something goes wrong. In this notebook you write each of them **by hand**, so that in the test you know what each line does and what happens when one is missing or misplaced.

---
## Theory in 60 seconds

- A tensor created with `requires_grad=True` is a **leaf**: PyTorch records every operation applied to it.
- `loss.backward()` walks the recorded graph backwards (chain rule) and stores `d loss / d leaf` in `leaf.grad`. `loss` must be a **scalar**.
- `.grad` is **accumulated** (added to), never overwritten. That is why it must be zeroed before the next `backward()`.
- A parameter update must not be recorded in the graph → it happens inside `with torch.no_grad():` and **in place** (`w -= ...`).
- Gradient descent: `w ← w − lr · ∂L/∂w`. The gradient points uphill; we step downhill. `lr` scales the step.

In [ ]:
import sys; sys.path.insert(0, "..")
import copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from dnd_utils import check

---
## Exercise 2.1 · Derivatives: by hand, then by autograd

$$f(a, b, c) = \frac{(a\,b + c)^2}{2}$$

**a)** Derive the three partial derivatives on paper and type them in as Python expressions of `a0, b0, c0`.

In [ ]:
a0, b0, c0 = 2.0, -3.0, 1.0

df_da = None
df_db = None
df_dc = None

**b)** Now let autograd do it: create three leaf tensors, compute `f`, call `backward()`, read the `.grad`s.

In [ ]:
a = None
b = None
c = None
# compute f and call backward here

In [ ]:
check("df/da", lambda: abs(df_da - a.grad.item()) < 1e-5 and abs(df_da - 15) < 1e-5)
check("df/db", lambda: abs(df_db - b.grad.item()) < 1e-5)
check("df/dc", lambda: abs(df_dc - c.grad.item()) < 1e-5)

---
## Exercise 2.2 · Gradients accumulate

Predict the three printed values **before** running the cell.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)

(w ** 2).backward()
first = w.grad.item()

(w ** 2).backward()            # a second backward WITHOUT zeroing
second = w.grad.item()

w.grad.zero_()
(5 * w).backward()
third = w.grad.item()

In [ ]:
pred_first, pred_second, pred_third = None, None, None

In [ ]:
check("first", lambda: pred_first == first)
check("second", lambda: pred_second == second)
check("third", lambda: pred_third == third)

What would a training loop do if you forgot `zero_grad()`? Describe the *symptom* you would see in the loss curve.

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 2.3 · The gradient of one neuron (the result behind logistic regression)

One neuron with a sigmoid and the binary cross-entropy loss, for a single sample `(x, y)`:

$$z = w\,x + b,\qquad p = \sigma(z) = \frac{1}{1+e^{-z}},\qquad L = -\big[y \ln p + (1-y)\ln(1-p)\big]$$

The famous result is $\dfrac{\partial L}{\partial w} = (p - y)\,x$ and $\dfrac{\partial L}{\partial b} = p - y$.

Verify it numerically: compute `L` with autograd, then compare `w.grad`, `b.grad` with the formula.

In [ ]:
x, y = 1.5, 1.0
w = torch.tensor(0.4, requires_grad=True)
b = torch.tensor(-0.2, requires_grad=True)

# forward + backward here (write the sigmoid and the loss yourself, no nn.* functions)
p = None

formula_dw = None
formula_db = None

In [ ]:
check("dL/dw = (p - y) x", lambda: abs(formula_dw - w.grad.item()) < 1e-6)
check("dL/db = (p - y)", lambda: abs(formula_db - b.grad.item()) < 1e-6)

Read the formula: the gradient is **(error) × (input)**. Three consequences worth remembering:

- a confident correct prediction (`p ≈ y`) produces almost no update;
- an input feature with large values produces large gradients for its weight → this is why unscaled features break gradient descent (notebook 03);
- the sign of `(p − y)` decides whether the weight goes up or down.

---
## Exercise 2.4 · Gradient descent on a parabola, and the learning rate

Minimise $f(w) = (w-3)^2 + 1$ starting from `w = 0`.

Write `descend(lr, steps)` that runs gradient descent **with autograd and a manual update** and returns the list of `w` values *after* each step (Python floats).

In [ ]:
def descend(lr, steps=20):
    w = torch.tensor(0.0, requires_grad=True)
    history = []
    # your loop
    return history

In [ ]:
check("lr=0.1: first step goes to 0.6", lambda: abs(descend(0.1)[0] - 0.6) < 1e-6)
check("lr=0.1: converges towards 3", lambda: abs(descend(0.1, 60)[-1] - 3) < 1e-3)
check("lr=0.5: jumps to the minimum in one step", lambda: abs(descend(0.5, 1)[0] - 3) < 1e-6)

Now **predict** what each learning rate does (`"converges"`, `"oscillates forever"`, or `"diverges"`), then look at the plot.

<details><summary>Hint — derive it instead of guessing</summary>

The gradient is `2(w − 3)`, so one step gives `w_new − 3 = (1 − 2·lr)·(w − 3)`. The distance to the minimum is multiplied by `(1 − 2·lr)` every step. What happens when the absolute value of that factor is below 1, exactly 1, above 1?
</details>

In [ ]:
behaviour = {0.1: None, 0.9: None, 1.0: None, 1.1: None}

In [ ]:
plt.figure(figsize=(7, 3))
for lr in behaviour:
    plt.plot(descend(lr, 15), "o-", ms=3, label=f"lr={lr}")
plt.axhline(3, color="gray", ls="--"); plt.ylim(-6, 12); plt.xlabel("step"); plt.ylabel("w"); plt.legend(); plt.show()

def _actual(lr):
    h = descend(lr, 200)
    if not h: return "?"
    return "converges" if abs(h[-1] - 3) < 1e-3 else ("diverges" if abs(h[-1] - 3) > 10 else "oscillates forever")
for lr, b_ in behaviour.items():
    check(f"lr={lr}: {_actual(lr)}", lambda: b_ == _actual(lr))

**Take-away.** "Too large a learning rate" is not vague folklore: each step multiplies the error by a factor that depends on `lr × curvature`. If the loss surface is much steeper in one direction than in another (unscaled features!), a learning rate that is fine for the flat direction diverges in the steep one.

---
## Exercise 2.5 · Linear regression with autograd, manual updates

Data: `y = 2.5·x − 1 + noise`. Fit `w` and `b` by full-batch gradient descent on the MSE. No `nn`, no `optim` — only tensors.

Record the loss **before** each update in `losses`.

In [ ]:
torch.manual_seed(0)
x = torch.linspace(-2, 2, 100)
y = 2.5 * x - 1 + 0.3 * torch.randn(100)

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr, losses = 0.1, []

for step in range(100):
    pass     # your four ideas: forward, backward, update (no_grad), zero

In [ ]:
check("100 losses recorded", lambda: len(losses) == 100)
check("loss decreased a lot", lambda: losses[-1] < 0.15 and losses[0] > 5)
check("w ≈ 2.5", lambda: abs(w.item() - 2.5) < 0.1)
check("b ≈ -1", lambda: abs(b.item() + 1) < 0.1)

The final loss is about `0.09`, not `0`. Why can it not go lower, no matter how long you train?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 2.6 · The same gradients, derived by hand in NumPy

For $L = \frac1n\sum_i (w x_i + b - y_i)^2$:

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_i (w x_i + b - y_i)\,x_i,\qquad \frac{\partial L}{\partial b} = \frac{2}{n}\sum_i (w x_i + b - y_i)$$

Implement `grads_numpy(w, b, x, y)` and check it against autograd at the point `w = 0.5, b = 0.2`.

In [ ]:
def grads_numpy(w, b, x, y):
    """x, y: NumPy arrays of shape (n,). Returns (dL/dw, dL/db) as floats."""
    raise NotImplementedError

In [ ]:
wt = torch.tensor(0.5, requires_grad=True); bt = torch.tensor(0.2, requires_grad=True)
((wt * x + bt - y) ** 2).mean().backward()
check("dL/dw matches autograd", lambda: abs(grads_numpy(0.5, 0.2, x.numpy(), y.numpy())[0] - wt.grad.item()) < 1e-4)
check("dL/db matches autograd", lambda: abs(grads_numpy(0.5, 0.2, x.numpy(), y.numpy())[1] - bt.grad.item()) < 1e-4)

---
## Exercise 2.7 · Bug hunt

Each of the three loops below is supposed to do the same as your loop in 2.5, and each contains **exactly one bug**. For each one: (1) predict the symptom, (2) run it, (3) fix it in the cell.

**Bug A**

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
try:
    for step in range(100):
        loss = ((w * x + b - y) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= 0.1 * w.grad
            b -= 0.1 * b.grad
    print("final loss", loss.item(), "w", w.item())
except Exception as e:
    print(type(e).__name__, e)
wA = w

**Bug B**

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
try:
    for step in range(100):
        loss = ((w * x + b - y) ** 2).mean()
        loss.backward()
        w = w - 0.1 * w.grad
        b = b - 0.1 * b.grad
    print("final loss", loss.item(), "w", w.item())
except Exception as e:
    print(type(e).__name__, e)
wB = w

**Bug C**

In [ ]:
w = torch.tensor(0.0)
b = torch.tensor(0.0)
try:
    for step in range(100):
        loss = ((w * x + b - y) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= 0.1 * w.grad
            b -= 0.1 * b.grad
        w.grad.zero_()
        b.grad.zero_()
    print("final loss", loss.item(), "w", w.item())
except Exception as e:
    print(type(e).__name__, e)
wC = w

In [ ]:
for name, ww in (("A", wA), ("B", wB), ("C", wC)):
    check(f"Bug {name} fixed: w ≈ 2.5", lambda: abs(ww.item() - 2.5) < 0.1)

---
## Exercise 2.8 · Write your own optimizer

`torch.optim.SGD` is not magic. Implement the two functions below so that they work for **any** list of parameters, then use them to train an `nn.Linear(1, 1)`.

<details><summary>Hint</summary>

`model.parameters()` yields the leaf tensors (weight and bias). A parameter that has not been through a `backward()` yet has `p.grad is None`.
</details>

In [ ]:
def my_zero_grad(params):
    raise NotImplementedError

def my_sgd_step(params, lr):
    raise NotImplementedError

The cell below trains two identical copies of a model: one with your functions, one with `torch.optim.SGD`. If your implementation is right, the two end up with **exactly** the same weights.

In [ ]:
torch.manual_seed(1)
X = x.reshape(-1, 1); Y = y.reshape(-1, 1)            # nn.Linear wants (n, 1); the target must match the output shape
model_mine = nn.Linear(1, 1)
model_torch = copy.deepcopy(model_mine)               # identical starting weights
opt = torch.optim.SGD(model_torch.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

try:
    for step in range(50):
        loss = loss_fn(model_mine(X), Y)
        my_zero_grad(list(model_mine.parameters()))
        loss.backward()
        my_sgd_step(list(model_mine.parameters()), 0.1)

        loss_t = loss_fn(model_torch(X), Y)
        opt.zero_grad()
        loss_t.backward()
        opt.step()
except NotImplementedError:
    pass
check("my optimizer == torch.optim.SGD (weight)", lambda: torch.allclose(model_mine.weight, model_torch.weight, atol=1e-6) and abs(model_torch.weight.item() - 2.5) < 0.1)
check("my optimizer == torch.optim.SGD (bias)", lambda: torch.allclose(model_mine.bias, model_torch.bias, atol=1e-6) and abs(model_torch.bias.item() + 1) < 0.1)

Now you know precisely what the four lines do:

| Line | What actually happens |
|---|---|
| `loss = loss_fn(model(X), Y)` | operations are recorded; `loss` is a scalar with a graph behind it |
| `opt.zero_grad()` | `p.grad` of every parameter is reset |
| `loss.backward()` | `p.grad` is filled with `∂loss/∂p` (added to what is there) |
| `opt.step()` | `p -= lr * p.grad` under `no_grad` (plain SGD; other optimizers use the gradient more cleverly) |

Does it matter whether `zero_grad()` is called *before the forward pass* or *between the forward pass and `backward()`*? What is the one place where it must **not** be?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Blank-page drill

Cover everything above. From memory, in the empty cell:

1. create `x` as 50 evenly spaced points in `[0, 1]` and `y = 4x + 2`;
2. fit `w` and `b` with **manual** gradient descent (autograd for the gradients, `lr = 0.5`, 500 steps);
3. print `w`, `b` and the final loss.

Expected: `w ≈ 4`, `b ≈ 2`, loss ≈ 0. If you had to look anything up, close the notebook and repeat tomorrow.

In [ ]:
# your code